# Week 8: Advanced ViT Optimizations for CIFAR-10

## Goal

Push wall-clock time to 94% further down by adding:
1. **`torch.compile`** — JIT kernel fusion, ~1.5-2x throughput on A100
2. **BF16 precision** — More stable than FP16 on A100, same speed
3. **Label smoothing** — Free ~0.5-1% accuracy gain
4. **Mixup augmentation** — Stronger regularization, faster convergence
5. **Gradient clipping** — Enables higher learning rates without instability
6. **Higher LR (1e-3)** — Converges faster with clipping + warmup
7. **AvgPool head** — Replace [CLS] token with mean pooling over all patches
8. **Muon optimizer** — Orthogonalized momentum for transformer weight matrices
9. **Ablation table** — Measure time-to-target for each optimization independently

Primary metric: **time-to-target** (seconds to reach 94% test accuracy).

## Imports and Setup

In [ ]:
import sys
import math
import time
import copy
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

from src.model import PatchEmbedding, TransformerBlock, count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

print(f"PyTorch: {torch.__version__}")
device = get_device()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

## New Components

### 1. ViT with Average Pooling Head

Replaces the `[CLS]` token with mean pooling over all patch tokens. One fewer learned parameter, often faster convergence on CIFAR-scale tasks.

In [ ]:
class ViTAvgPool(nn.Module):
    """
    ViT with global average pooling instead of [CLS] token.
    Eliminates CLS token overhead and often converges faster on small datasets.
    """

    def __init__(
        self,
        img_size: int = 32,
        patch_size: int = 4,
        in_channels: int = 3,
        num_classes: int = 10,
        embed_dim: int = 192,
        depth: int = 6,
        num_heads: int = 6,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim,
        )
        n_patches = self.patch_embed.n_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)        # (B, n_patches, embed_dim)
        x = x + self.pos_embed
        x = self.pos_dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x.mean(dim=1)              # global average pool over patches
        return self.head(x)

### 2. Muon Optimizer

Inspired by [modded-nanoGPT](https://github.com/KellerJordan/modded-nanogpt). Applies Nesterov momentum with Newton-Schulz orthogonalization to weight matrix gradients. Empirically finds better descent directions for transformer matrices than AdamW.

- 2D weight matrices (attention, MLP) → Muon update with orthogonalization
- 1D params (biases, LayerNorm, embeddings) → AdamW update

In [ ]:
def _zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Newton-Schulz iteration to orthogonalize gradient matrix G."""
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.to(torch.float32)
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X


class Muon(torch.optim.Optimizer):
    """
    Muon optimizer for transformer weight matrices.
    Uses Nesterov momentum + Newton-Schulz orthogonalization.
    Only applied to 2D weight matrices; other params use AdamW.
    """

    def __init__(self, params, lr: float = 0.02, momentum: float = 0.95, ns_steps: int = 5):
        defaults = dict(lr=lr, momentum=momentum, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            ns_steps = group["ns_steps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if "buf" not in state:
                    state["buf"] = torch.zeros_like(g)
                buf = state["buf"]
                buf.mul_(momentum).add_(g)
                # Nesterov: look-ahead gradient
                g = g + momentum * buf
                if g.ndim >= 2:
                    g = _zeropower_via_newtonschulz5(g, steps=ns_steps).to(p.dtype)
                    g = g * (max(g.size(0), g.size(1)) ** 0.5)
                p.add_(g, alpha=-lr)


def get_muon_optimizer(model: nn.Module, muon_lr: float = 0.02, adamw_lr: float = 3e-4):
    """
    Split model params: 2D weight matrices -> Muon, everything else -> AdamW.
    Returns a list of optimizers to step together.
    """
    muon_params, adamw_params = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            if p.ndim >= 2 and "embed" not in name and "head" not in name:
                muon_params.append(p)
            else:
                adamw_params.append(p)
    muon = Muon(muon_params, lr=muon_lr)
    adamw = torch.optim.AdamW(adamw_params, lr=adamw_lr, weight_decay=0.01)
    return muon, adamw


print("Muon optimizer defined.")

### 3. Mixup Augmentation

In [ ]:
def mixup_batch(
    data: torch.Tensor, target: torch.Tensor, alpha: float = 0.4
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Apply Mixup to a batch. Returns mixed data, target_a, target_b, lambda."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(data.size(0), device=data.device)
    mixed = lam * data + (1 - lam) * data[idx]
    return mixed, target, target[idx], lam


def mixup_criterion(
    criterion: nn.Module,
    output: torch.Tensor,
    target_a: torch.Tensor,
    target_b: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    return lam * criterion(output, target_a) + (1 - lam) * criterion(output, target_b)


print("Mixup helpers defined.")

## Configuration

In [ ]:
SEED = 42
TARGET_ACCURACY = 94.0
MAX_EPOCHS = 100

DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)

# Data loading (A100 defaults)
BATCH_SIZE = 512
NUM_WORKERS = 4
PERSISTENT_WORKERS = True

# Toggle each optimization for ablation
USE_COMPILE     = True   # torch.compile
USE_BF16        = True   # BF16 instead of FP16
USE_LABEL_SMOOTH = True  # label smoothing = 0.1
USE_MIXUP       = True   # Mixup alpha=0.4
USE_GRAD_CLIP   = True   # clip_grad_norm = 1.0
USE_HIGH_LR     = True   # LR = 1e-3 instead of 3e-4
USE_AVGPOOL     = True   # AvgPool head instead of CLS token
USE_MUON        = True   # Muon optimizer for weight matrices
USE_SCHEDULER   = True
WARMUP_EPOCHS   = 5

LR = 1e-3 if USE_HIGH_LR else 3e-4

print(f"Device: {device} | BF16: {USE_BF16} | Compile: {USE_COMPILE}")
print(f"LR: {LR} | Mixup: {USE_MIXUP} | Muon: {USE_MUON} | AvgPool: {USE_AVGPOOL}")

## Data Loading

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape:   {sample_batch.shape}")

## Model Creation

In [ ]:
def make_model(use_avgpool: bool = True) -> nn.Module:
    if use_avgpool:
        return ViTAvgPool(
            img_size=32, patch_size=4, embed_dim=192,
            depth=6, num_heads=6, dropout=0.0,
        )
    from src.utils import get_model
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)


model = make_model(use_avgpool=USE_AVGPOOL)
print(f"Model: {'AvgPool' if USE_AVGPOOL else 'CLS-token'} ViT")
print(f"Parameters: {count_parameters(model):,}")

reset_peak_gpu_memory()
model = model.to(device)
_ = model(sample_batch.to(device))
print(f"Output shape: {_.shape}")

## Custom Training Loop

Incorporates all Week 8 optimizations: `torch.compile`, BF16 autocast, label smoothing, Mixup, gradient clipping, and Muon.

In [ ]:
def train_w8(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    num_epochs: int = MAX_EPOCHS,
    lr: float = LR,
    device: torch.device = device,
    target_accuracy: float = TARGET_ACCURACY,
    use_compile: bool = USE_COMPILE,
    use_bf16: bool = USE_BF16,
    use_label_smooth: bool = USE_LABEL_SMOOTH,
    use_mixup: bool = USE_MIXUP,
    use_grad_clip: bool = USE_GRAD_CLIP,
    use_muon: bool = USE_MUON,
    use_scheduler: bool = USE_SCHEDULER,
    warmup_epochs: int = WARMUP_EPOCHS,
    log_every: int = 1,
) -> Dict[str, Any]:

    model = model.to(device)

    if use_compile and device.type == "cuda":
        print("Compiling model with torch.compile...")
        model = torch.compile(model)

    label_smooth = 0.1 if use_label_smooth else 0.0
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)

    if use_muon and device.type == "cuda":
        muon_opt, adamw_opt = get_muon_optimizer(model, muon_lr=lr * 20, adamw_lr=lr)
        optimizers = [muon_opt, adamw_opt]
    else:
        adamw_opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        optimizers = [adamw_opt]

    amp_enabled = device.type == "cuda"
    amp_dtype = torch.bfloat16 if (use_bf16 and torch.cuda.is_bf16_supported()) else torch.float16
    scaler = torch.amp.GradScaler("cuda") if (amp_enabled and amp_dtype == torch.float16) else None

    schedulers = []
    if use_scheduler:
        for opt in optimizers:
            def lr_lambda(ep, _warmup=warmup_epochs, _total=num_epochs):
                if ep < _warmup:
                    return (ep + 1) / _warmup
                progress = (ep - _warmup) / max(1, _total - _warmup)
                return 0.5 * (1 + math.cos(math.pi * progress))
            schedulers.append(torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda))

    history = {"train_loss": [], "train_acc": [], "test_loss": [],
               "test_acc": [], "throughput": [], "wall_time": []}
    best_acc = 0.0
    time_to_target = None
    total_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        epoch_start = time.perf_counter()

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            for opt in optimizers:
                opt.zero_grad(set_to_none=True)

            if use_mixup:
                data, target_a, target_b, lam = mixup_batch(data, target)

            if amp_enabled:
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    output = model(data)
                    if use_mixup:
                        loss = mixup_criterion(criterion, output, target_a, target_b, lam)
                    else:
                        loss = criterion(output, target)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    if use_grad_clip:
                        scaler.unscale_(optimizers[-1])
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    for opt in optimizers:
                        scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    if use_grad_clip:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    for opt in optimizers:
                        opt.step()
            else:
                output = model(data)
                if use_mixup:
                    loss = mixup_criterion(criterion, output, target_a, target_b, lam)
                else:
                    loss = criterion(output, target)
                loss.backward()
                if use_grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                for opt in optimizers:
                    opt.step()

            total_loss += loss.item()
            pred = output.argmax(dim=1)
            ref = target if not use_mixup else target_a
            correct += pred.eq(ref).sum().item()
            total += ref.size(0)

        for sched in schedulers:
            sched.step()

        epoch_time = time.perf_counter() - epoch_start
        throughput = total / epoch_time
        avg_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        test_loss, test_acc = validate(model, test_loader, nn.CrossEntropyLoss(), device)
        best_acc = max(best_acc, test_acc)

        if time_to_target is None and test_acc >= target_accuracy:
            time_to_target = time.perf_counter() - total_start

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["throughput"].append(throughput)
        history["wall_time"].append(epoch_time)

        if log_every > 0 and epoch % log_every == 0:
            mem = get_peak_gpu_memory_mb()
            mem_str = f"{mem:.0f} MB" if mem else "N/A"
            ttt_str = f" [HIT @ {time_to_target:.1f}s]" if time_to_target else ""
            print(
                f"Epoch {epoch:3d}/{num_epochs} | "
                f"Loss: {avg_loss:.4f} Acc: {train_acc:.1f}% | "
                f"Test: {test_acc:.2f}% | "
                f"{throughput:.0f} img/s | Mem: {mem_str}{ttt_str}"
            )

        if time_to_target is not None:
            print(f"\nTarget {target_accuracy}% reached at epoch {epoch}. Time: {time_to_target:.2f}s")
            break

    total_time = time.perf_counter() - total_start
    return {
        "history": history,
        "best_acc": best_acc,
        "total_time": total_time,
        "time_to_target": time_to_target,
        "peak_gpu_mb": get_peak_gpu_memory_mb(),
    }

## Run: Fully Optimized Config

In [ ]:
set_seed(SEED)
reset_peak_gpu_memory()
model = make_model(use_avgpool=USE_AVGPOOL).to(device)

results = train_w8(model, train_loader, test_loader)

ttt = results.get("time_to_target")
print("\n--- Summary ---")
print(f"Best test accuracy:         {results['best_acc']:.2f}%")
print(f"Time-to-{TARGET_ACCURACY}%: {ttt:.2f}s" if ttt else f"Target not reached")
print(f"Total wall-clock:           {results['total_time']:.2f}s")
print(f"Peak GPU memory:            {results.get('peak_gpu_mb', 'N/A')} MB")